# Visualizations for the three moe model in test_three_moe.py

In [ ]:
# Import statements
import matplotlib.pyplot as plt
import torch
from train_moe import train_moe_model
from three_moe_helpers import (
    make_three_region_data,
    make_three_expert_problem,
    gating_weights,
)
from fractions import Fraction

In [ ]:
# Set constants and random seed
NUM_EXPERTS = 3

In [ ]:
# Generate synthetic data
X, Y, region, edges = make_three_region_data()

### The Synthetic Data

In [ ]:
_, ax = plt.subplots()
ax.scatter(X, Y)
ax.set_xlabel("x")
ax.set_ylabel("y")
plt.tight_layout()

In [ ]:
# Create stochastic three-expert model
surrogate_model, model, X, Y, region, edges = make_three_expert_problem()

print(f"Surrogate model has {surrogate_model.num_params()} parameters")
print(f"Created stochastic model with {model.num_params()} parameters")

In [ ]:
# Train the stochastic model
total_loss = train_moe_model(surrogate_model, model, X, Y)

print(f"Final total loss = {total_loss:.4f}")

In [ ]:
# Evaluate the model
model.eval()
with torch.no_grad():
    X_test = torch.linspace(-1, 1, 300).unsqueeze(1)
    Y_test = model(X_test, num_samples=1000)

### Data and Model

Plot the model's mean prediction on the same graph as the generated data.

Note that the 80th percentile prediction is also plotted, but the width is small enough that the uncertainty is not shown on the plot.

In [ ]:
# Plot prediction
_, ax = plt.subplots()
Q_test = torch.quantile(Y_test, torch.tensor([0.1, 0.9]), axis=0)
X_plot = X_test.squeeze(-1)
Q_plot = Q_test.squeeze(-1)
ax.fill_between(X_plot, Q_plot[0], Q_plot[-1], color="red", alpha=0.5, linewidth=0)
M_plot = Y_test.mean(axis=0).squeeze(-1)
ax.plot(X_plot, M_plot, color="red", linewidth=2, label="MoE mean")
ax.scatter(X, Y, zorder=99)
for edge in edges[1:-1]:
    ax.axvline(edge.item(), color="black", linestyle="--", linewidth=1)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend(frameon=False)
plt.tight_layout()

In [ ]:
# Evaluate the gating weights
with torch.no_grad():
    params = model.sample_parameters(1000)
    gating_weights = surrogate_model.get_gating_weights(X_test, params)
    expert_outputs = surrogate_model.get_expert_outputs(X_test, params)

### Experts and Data

This plot shows how each expert fits the data.

In [ ]:
# Plot expert outputs
_, ax = plt.subplots()
ax.scatter(X, Y, zorder=99)
for expert in range(NUM_EXPERTS):
    Q_test = torch.quantile(
        expert_outputs[:, expert, :, :],
        torch.tensor([0.1, 0.5, 0.9]),
        axis=0,
    )
    X_plot = X_test.squeeze(-1)
    Q_plot = Q_test.squeeze(-1)
    ax.fill_between(
        X_plot,
        Q_plot[0],
        Q_plot[-1],
        color=f"C{expert + 1}",
        alpha=0.5,
        linewidth=0,
    )
    M_plot = expert_outputs[:, expert, :, :].mean(axis=0).squeeze(-1)
    ax.plot(
        X_plot,
        M_plot,
        color=f"C{expert + 1}",
        linewidth=2,
        label=f"expert {expert + 1}",
    )
for edge in edges[1:-1]:
    ax.axvline(edge.item(), color="black", linestyle="--", linewidth=1)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend(frameon=False)
plt.tight_layout()

### Gating Weights

Sample the model to determine the weight for each expert over the sample space of x values. The plots show each expert is dominant in a unique region.

In [ ]:
# Plot gating weights
_, ax = plt.subplots()
for expert in range(NUM_EXPERTS):
    Q_test = torch.quantile(
        gating_weights[:, :, expert],
        torch.tensor([0.1, 0.5, 0.9]),
        axis=0,
    )
    X_plot = X_test.squeeze(-1)
    Q_plot = Q_test.squeeze(-1)
    ax.fill_between(
        X_plot,
        Q_plot[0],
        Q_plot[-1],
        color=f"C{expert + 1}",
        alpha=0.5,
        linewidth=0,
    )
    ax.plot(
        X_plot,
        Q_plot[1],
        color=f"C{expert + 1}",
        linewidth=2,
        label=f"expert {expert + 1}",
    )

for edge in edges[1:-1]:
    ax.axvline(edge.item(), color="black", linestyle="--", linewidth=1)

ax.set_xlabel("x")
ax.set_ylabel("expert mixing weight")
ax.legend(frameon=False)
plt.tight_layout()

In [ ]:
# Region-wise average gate weights
X_grid = torch.linspace(-1.0, 1.0, 600).unsqueeze(-1)

grid_region = torch.bucketize(
    X_grid.squeeze(-1),
    edges[1:-1],
)

with torch.no_grad():
    params = model.sample_parameters(num_samples=200)
    gates = surrogate_model.get_gating_weights(X_grid, params)
    mean_gates = gates.mean(dim=0)

# Average gate vector in each of the three regions.
region_gate_means = torch.stack(
    [mean_gates[grid_region == r].mean(dim=0) for r in range(NUM_EXPERTS)]
)

dominant_expert_by_region = region_gate_means.argmax(dim=1)
dominant_weights = region_gate_means.max(dim=1).values


def format_edge(x):
    return str(Fraction(float(x)).limit_denominator())


region_labels = []
for r in range(NUM_EXPERTS):
    left = format_edge(edges[r])
    right = format_edge(edges[r + 1])

    if r < NUM_EXPERTS - 1:
        region_labels.append(f"[{left}, {right})")
    else:
        region_labels.append(f"[{left}, {right}]")


# Plot gate weights
# Transpose so rows are experts and columns are regions.
_, ax = plt.subplots()
im = ax.imshow(region_gate_means.T, vmin=0.0, vmax=1.0, cmap="viridis")

ax.set_xlabel("region")
ax.set_ylabel("expert")

ax.set_xticks(range(NUM_EXPERTS))
ax.set_xticklabels(region_labels)

ax.set_yticks(range(NUM_EXPERTS))
ax.set_yticklabels([1, 2, 3])

for r in range(NUM_EXPERTS):
    for e in range(NUM_EXPERTS):
        ax.text(
            r,
            e,
            f"{region_gate_means[r, e]:.2f}",
            ha="center",
            va="center",
            color="red",
        )

plt.colorbar(im, ax=ax, label="mean gating weight")
plt.tight_layout()